## Power Iteration - CuPy - Memory Spaces

### Table of Contents
1. [Introduction to Memory Spaces](#1-introduction-to-memory-spaces)
2. [The CPU Baseline (NumPy)](#2-the-cpu-baseline-numpy)
3. [The GPU Port (CuPy)](#3-the-gpu-port-cupy)
4. [Optimizing Data Generation](#4-optimizing-data-generation)
5. [Verification and Benchmarking](#5-verification-and-benchmarking)
6. [Extra Credit](#extra-credit)

---

### 1. Introduction to Memory Spaces

Before we implement algorithms on the GPU, we must understand the hardware architecture. A heterogeneous system (like the one you are using) consists of two memory spaces: 

1. **Host Memory:** Accessible by the CPU.
2. **Device Memory:** Accessible by the GPU.

To ensure data is accessible from a particular processor, we need to explicity transfer it:

* **Host $\to$ Device:** Move data to the GPU to compute.
    * Syntax: `x_device = cp.asarray(x_host)`
* **Device $\to$ Host:** Move results back to the CPU to save to disk, plot with Matplotlib, or print.
    * Syntax: `y_host = cp.asnumpy(y_device)`

#### Implicit Transfers and Synchronization

It is crucial to understand when CuPy interacts with the CPU implicitly. These interactions can kill performance because they force the GPU to pause (synchronize) while data moves.

CuPy silently transfers and synchronizes when you:
1.  **Print** a GPU array (`print(gpu_array)`).
2.  **Convert** to a Python scalar (`float(gpu_array)` or `.item()`).
3.  **Evaluate** a GPU scalar in a boolean context (`if gpu_scalar > 0:`).

#### The Task
To understand the implications of these concepts, let's experiment with estimating the dominant eigenvalue of a matrix using the **Power Iteration** algorithm.

Before we dive into the code, let's understand the math behind the algorithm we are implementing.

**Power Iteration** is a classic iterative method used to find the dominant eigenvalue (the eigenvalue with the largest absolute value) and its corresponding eigenvector of a square matrix $A$.

##### How It Works

The core idea is simple: if you repeatedly multiply a vector by a matrix $A$, the vector will eventually converge towards the dominant eigenvector of $A$, regardless of the initial vector you started with (provided the initial vector has some component in the direction of the dominant eigenvector).

##### The Mathematical Steps

Given a square matrix $A$ and a random initial vector $x_0$, the algorithm proceeds as follows for each step $k$:

**1. Matrix-Vector Multiplication:**

We calculate the next approximation of the vector:

$$y = A x_k$$

**2. Eigenvalue Estimation (Rayleigh Quotient):**

We estimate the eigenvalue $\lambda$ using the current vector. This is essentially projecting $y$ onto $x$:

$$\lambda_k = \frac{x_k^T y}{x_k^T x_k} = \frac{x_k^T A x_k}{x_k^T x_k}$$

**3. Residual Calculation (Error Check):**

We check how close we are to the true definition of an eigenvector ($Ax = \lambda x$) by calculating the "residual" (error):

$$r = ||y - \lambda_k x_k||$$

If $r$ is close to 0, we have converged.

**4. Normalization:**

To prevent the numbers from exploding (overflow) or vanishing (underflow), we normalize the vector for the next iteration:

$$x_{k+1} = \frac{y}{||y||}$$

We will start with a standard CPU implementation, port it to the GPU using CuPy, and analyze the performance impact of memory transfers.

In [1]:
import numpy as np
import cupy as cp
import cupyx as cpx
import time
from dataclasses import dataclass

# Configuration for the algorithm
@dataclass
class PowerIterationConfig:
    dim: int = 10000                   # Matrix size (dim x dim)
    dominance: float = 0.05            # How much larger the top eigenvalue is (controls convergence, higher == faster)
    max_steps: int = 1000              # Maximum iterations
    check_frequency: int = 25          # Check for convergence every N steps
    progress: bool = True              # Print progress logs
    residual_threshold: float = 1e-10  # Stop if error is below this

### 2. The CPU Baseline (NumPy)

We generate a random dense matrix that is diagonalizable. This data is generated on the **Host (CPU)** and resides in **Host Memory**.


In [2]:
def generate_host(cfg=PowerIterationConfig()):
    """Generates a random diagonalizable matrix on the CPU."""
    np.random.seed(42)

    # Create eigenvalues: One large one (1.0), the rest smaller
    weak_lam = np.random.random(cfg.dim - 1) * (1.0 - cfg.dominance)
    lam = np.random.permutation(np.concatenate(([1.0], weak_lam)))

    # Construct matrix A = P * D * P^-1
    P = np.random.random((cfg.dim, cfg.dim))  # Random invertible matrix
    D = np.diag(np.random.permutation(lam))   # Diagonal matrix of eigenvalues
    A = ((P @ D) @ np.linalg.inv(P))          # The final matrix
    return A

# Generate the data on Host
print("Generating Host Data...")
A_host = generate_host()
print(f"Host Matrix Shape: {A_host.shape}")
print(f"Data Type: {A_host.dtype}")

Generating Host Data...
Host Matrix Shape: (10000, 10000)
Data Type: float64


#### Implementing Power Iteration (CPU)

As described above, the Power Iteration algorithm repeatedly multiplies a vector $x$ by matrix $A$ ($y = Ax$) and normalizes the result. We initialize this algorithm with a vector of 1s ($x_0$) as our initial guess.

In [3]:
def estimate_host(A, cfg=PowerIterationConfig()) -> np.ndarray:
    """
    Performs power iteration using purely NumPy (CPU).
    """
    # Initialize solution vector.
    x = np.ones(A.shape[0], dtype=np.float64)

    for i in range(0, cfg.max_steps, cfg.check_frequency):
        # Matrix-Vector multiplication.
        y = A @ x

        # Rayleigh quotient.
        lam = (x @ y) / (x @ x)

        # Calculate residual (error).
        res = np.linalg.norm(y - lam * x)

        # Normalize vector for next step.
        x = y / np.linalg.norm(y)

        if cfg.progress:
            print(f"Step {i}: residual = {res:.3e}")

        # Save a checkpoint.
        np.savetxt(f"/tmp/host_{i}.txt", x)

        # Convergence check.
        if res < cfg.residual_threshold:
            break

        # Run intermediate steps without checking residual to save compute.
        for _ in range(i + 1, min(i + cfg.check_frequency, cfg.max_steps)):
            y = A @ x
            x = y / np.linalg.norm(y)

    return (x.T @ (A @ x)) / (x.T @ x)

lam_est_host = estimate_host(A_host)

assert isinstance(lam_est_host, (np.ndarray, np.generic)), "Must return a NumPy array or NumPy scalar"
np.testing.assert_allclose(lam_est_host, 1, atol=1e-4)

print()
print("Dominant Eigenvalue:", lam_est_host)

Step 0: residual = 8.634e+00
Step 25: residual = 5.237e-03
Step 50: residual = 6.719e-03
Step 75: residual = 8.694e-03
Step 100: residual = 4.068e-03
Step 125: residual = 1.165e-03
Step 150: residual = 2.956e-04
Step 175: residual = 7.372e-05
Step 200: residual = 1.848e-05
Step 225: residual = 4.669e-06
Step 250: residual = 1.189e-06
Step 275: residual = 3.046e-07
Step 300: residual = 7.847e-08
Step 325: residual = 2.031e-08
Step 350: residual = 5.278e-09
Step 375: residual = 1.376e-09
Step 400: residual = 3.600e-10
Step 425: residual = 9.280e-11

Dominant Eigenvalue: 0.999999999902756


### 3. The GPU Port (CuPy)

#### Exercise: Port the CPU Implementation to GPU

Now it's your turn! Your task is to convert the `estimate_host` function to run on the GPU using CuPy.

**Remember the rules of Memory Spaces:**
1.  **Transfer:** Move `A_host` from CPU to GPU using `cp.asarray()`.
2.  **Compute:** Perform math using `cp` functions on the GPU.
3.  **Retrieve:** Move result back to CPU using `cp.asnumpy()`.

**Hint:** CuPy tries to replicate the NumPy API. In many cases, you can simply change `np.` to `cp.`. However, CuPy operations *must* run on data present in Device Memory.

**The code below starts as a copy of the CPU implementation. Modify it to run on the GPU:**


In [5]:
def estimate_device(A, cfg=PowerIterationConfig()) -> np.ndarray:
    """
    TODO: Port the power iteration algorithm to the GPU using CuPy.

    Steps to complete:
    1. Transfer the input matrix A to the GPU
    2. Initialize the vector x on the GPU
    3. Replace np operations with cp operations
    4. Copy x from device to host and save a checkpoint
    5. Return the result as a NumPy array
    """
    # Initialize solution vector
    x_gpu = cp.ones(A.shape[0], dtype=cp.float64)

    A_GPU = cp.asarray(A)

    for i in range(0, cfg.max_steps, cfg.check_frequency):
        # Matrix-Vector multiplication
        y = A_GPU @ x_gpu

        # Rayleigh quotient: (x . y) / (x . x)
        lam = (x_gpu @ y) / (x_gpu @ x_gpu)

        # Calculate residual (error)
        res = cp.linalg.norm(y - lam * x_gpu)

        # Normalize vector for next step
        x_gpu = y / cp.linalg.norm(y)

        if cfg.progress:
            print(f"Step {i}: residual = {res:.3e}")

        cp.savetxt(f"/tmp/host_{i}.txt", x_gpu) # Save a checkpoint.

        # Convergence check
        if res < cfg.residual_threshold:
            break

        # Run intermediate steps without checking residual to save compute
        for _ in range(i + 1, min(i + cfg.check_frequency, cfg.max_steps)):
            y = A_GPU @ x_gpu
            x_gpu = y / cp.linalg.norm(y)

    x_gpu = (x_gpu.T @ (A_GPU @ x_gpu)) / (x_gpu.T @ x_gpu)
    
    x = cp.asnumpy(x_gpu)

    return x

lam_est_device = estimate_device(A_host)

assert isinstance(lam_est_device, (np.ndarray, np.generic)), "Must return a NumPy array or NumPy scalar"
np.testing.assert_allclose(lam_est_device, 1, atol=1e-4)

print()
print("Dominant Eigenvalue:", lam_est_device)

Step 0: residual = 8.634e+00
Step 25: residual = 5.237e-03
Step 50: residual = 6.719e-03
Step 75: residual = 8.694e-03
Step 100: residual = 4.068e-03
Step 125: residual = 1.165e-03
Step 150: residual = 2.956e-04
Step 175: residual = 7.372e-05
Step 200: residual = 1.848e-05
Step 225: residual = 4.669e-06
Step 250: residual = 1.189e-06
Step 275: residual = 3.046e-07
Step 300: residual = 7.847e-08
Step 325: residual = 2.031e-08
Step 350: residual = 5.278e-09
Step 375: residual = 1.376e-09
Step 400: residual = 3.597e-10
Step 425: residual = 9.470e-11

Dominant Eigenvalue: 0.9999999999028775


### 4. Optimizing Data Generation

In the previous step, we generated data on the CPU and copied it to the GPU. For large datasets, the transfer time from host to device can be a bottleneck. 

It is almost always faster to **generate** the data directly on the GPU if possible.

#### Exercise: Generate Data Directly on the GPU

Your task is to convert the `generate_host` function to generate the matrix directly on the GPU using CuPy's random functions.

**Hints:**
- Use `cp.random.seed()` instead of `np.random.seed()`
- Use `cp.random.random()` instead of `np.random.random()`
- Use `cp.random.permutation()` instead of `np.random.permutation()`
- Use `cp.concatenate()`, `cp.array()`, `cp.diag()`, and `cp.linalg.inv()`

**The code below starts as a copy of the CPU implementation. Modify it to generate data directly on the GPU:**


In [7]:
def generate_device(cfg=PowerIterationConfig()):
    """
    TODO: Port this CPU implementation to generate data directly on the GPU.
    Replace all np operations with their cp equivalents:
    - np.random.seed -> cp.random.seed
    - np.random.random -> cp.random.random
    - np.random.permutation -> cp.random.permutation
    - np.concatenate -> cp.concatenate
    - np.diag -> cp.diag
    - np.linalg.inv -> cp.linalg.inv
    """
    cp.random.seed(42)

    # Create eigenvalues: One large one (1.0), the rest smaller
    weak_lam = cp.random.random(cfg.dim - 1) * (1.0 - cfg.dominance)
    lam = cp.random.permutation(cp.concatenate((cp.array([1.0]), weak_lam)))

    # Construct matrix A = P * D * P^-1
    P = cp.random.random((cfg.dim, cfg.dim))
    D = cp.diag(cp.random.permutation(lam))
    A = ((P @ D) @ cp.linalg.inv(P))
    return A

A_device = generate_device()

lam_est_device_gen = estimate_device(A_device)

np.testing.assert_allclose(lam_est_device_gen, 1, atol=1e-4)

print()
print("Dominant Eigenvalue:", lam_est_device_gen)

Step 0: residual = 2.184e+01
Step 25: residual = 1.338e-02
Step 50: residual = 7.808e-02
Step 75: residual = 1.423e-02
Step 100: residual = 2.416e-03
Step 125: residual = 5.371e-04
Step 150: residual = 1.275e-04
Step 175: residual = 3.080e-05
Step 200: residual = 7.500e-06
Step 225: residual = 1.836e-06
Step 250: residual = 4.518e-07
Step 275: residual = 1.118e-07
Step 300: residual = 2.785e-08
Step 325: residual = 6.981e-09
Step 350: residual = 1.762e-09
Step 375: residual = 4.483e-10
Step 400: residual = 1.147e-10
Step 425: residual = 2.958e-11

Dominant Eigenvalue: 1.0000000000209366


#### Think About It

Both functions use `seed(42)`. Are `A_host` and `A_device` identical? Try comparing them:

In [8]:
with np.printoptions(precision=4):
    print("A_host:")
    print(A_host)
    print()
    print("A_device:")
    print(A_device)
    print()

A_host:
[[ 0.7063 -0.946   0.8389 ...  0.411   0.206   0.4597]
 [ 0.4194 -0.72    0.3512 ... -0.1365 -0.297   0.3771]
 [ 0.1829 -0.1875  0.9952 ...  0.2476  0.2321  0.3831]
 ...
 [ 0.3175 -0.8111  0.3791 ...  0.3407 -0.1774  0.3547]
 [ 0.3982 -0.2501  0.4781 ... -0.2793  0.5011  0.2738]
 [ 0.1355 -0.5441  0.913  ...  0.0292 -0.0937  0.8388]]

A_device:
[[ 0.0058  0.1203  0.2237 ... -0.0776  0.3863  0.3676]
 [ 0.0738  0.3409 -0.2932 ...  0.0479 -0.1212  0.2116]
 [-0.2237 -0.1585  0.3557 ... -0.0457 -0.0158  0.1588]
 ...
 [-0.0306 -0.3322 -0.5239 ...  0.0808  0.1574  0.3686]
 [-0.3641  0.0413  0.2688 ... -0.0943  0.7714  0.2987]
 [-0.3491 -0.0357  0.2387 ... -0.2674  0.393   0.737 ]]



What does this reveal about `np.random` vs `cp.random`?

### 5. Verification and Benchmarking

Finally, let's verify our accuracy against a reference implementation (`numpy.linalg.eigvals()`) and benchmark the speedup.


In [9]:
start = time.perf_counter()
lam_ref = np.linalg.eigvals(A_host).real.max()
T_ref = time.perf_counter() - start

In [10]:
print(f"Power Iteration (Host)   = {lam_est_host}")
print(f"Power Iteration (Device) = {lam_est_device}")
print(f"`eigvals` Reference      = {lam_ref}")

rel_err_host   = abs(lam_est_host - lam_ref) / abs(lam_ref)
rel_err_device = abs(lam_est_device - lam_ref) / abs(lam_ref)
print()
print(f"Relative Error (Host)    = {rel_err_host:.3e}")
print(f"Relative Error (Device)  = {rel_err_device:.3e}")

np.testing.assert_allclose(lam_est_host, lam_ref, rtol=1e-4)
np.testing.assert_allclose(lam_est_device, lam_ref, rtol=1e-4)

Power Iteration (Host)   = 0.999999999902756
Power Iteration (Device) = 0.9999999999028775
`eigvals` Reference      = 1.0000000000005147

Relative Error (Host)    = 9.776e-11
Relative Error (Device)  = 9.764e-11


#### Benchmarking with `cupyx.profiler.benchmark()`

We use CuPy's built-in benchmarking utility for accurate GPU timing. This handles warmup and synchronization automatically.

We intentionally use `A_host` for both benchmarks, not `A_device`, because they're not the same matrices due to differences in NumPy and CuPy's random facilities. Different matrices converge at different rates, so it's only valid to benchmark on the same inputs.

In [11]:
cfg = PowerIterationConfig(progress=False)

print("Timing Host...")
T_host = cpx.profiler.benchmark(estimate_host, args=(A_host, cfg), n_repeat=10, n_warmup=1).cpu_times

print("Timing Device...")
T_device = cpx.profiler.benchmark(estimate_device, args=(A_host, cfg), n_repeat=10, n_warmup=1).cpu_times

print()
print(f"Power Iteration (Host)   = {T_host.mean() * 1000:.6g} ms ± {(T_host.std() / T_host.mean()):.2%} (mean ± relative stdev of {T_host.size} runs)")
print(f"Power Iteration (Device) = {T_device.mean() * 1000:.6g} ms ± {(T_device.std() / T_device.mean()):.2%} (mean ± relative stdev of {T_device.size} runs)")
print(f"`eigvals` Reference      = {T_ref * 1000:.6g} ms")

speedup = T_host.mean() / T_device.mean()
print()
print(f"Speedup (Device over Host) = {speedup:.1f}x")

Timing Host...
Timing Device...

Power Iteration (Host)   = 669.013 ms ± 0.22% (mean ± relative stdev of 10 runs)
Power Iteration (Device) = 343.13 ms ± 0.09% (mean ± relative stdev of 10 runs)
`eigvals` Reference      = 45512.1 ms

Speedup (Device over Host) = 1.9x


---

### Extra Credit

**Explore the impact of changing the following parameters:**

1. **Problem Size (`dim`):** How does the GPU speedup change as you decrease the matrix dimensions? Try values like 1024, 2048, 4096, 8192.

2. **Compute Workload (`max_steps` and `dominance`):** The `dominance` parameter controls how quickly the algorithm converges. A smaller dominance means eigenvalues are closer together, requiring more iterations. How does this affect the CPU vs GPU comparison?

3. **Check Frequency (`check_frequency`):** This controls how often we check for convergence (and trigger implicit CPU synchronization via the print statement). What happens to GPU performance when you check every step (`check_frequency=1`) vs. less frequently (`check_frequency=50`)?

**Experiment below:**

In [ ]:
# Try different configurations here!
# Example:
# cfg_smaller = PowerIterationConfig(dim=8192, progress=False)
# cfg_slow_converge = PowerIterationConfig(dominance=0.01, progress=False)
# cfg_frequent_check = PowerIterationConfig(check_frequency=1, progress=True)

# Your experiments: